In [52]:
import os, sys
from pathlib import Path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
import numpy as np
import pandas as pd
import data_io as f1  
import series_utils as f2
import portfolio
import importlib
import config
import risk_matrix
import books

importlib.reload(books)
importlib.reload(risk_matrix)
importlib.reload(f1)
importlib.reload(f2)

# print('params:\n', config.params)
window_start="2025-4-10"
holdings=books.IBKR_live
ohlc = False

print('max age:', config.params['max_age'])

print('---- Building returns matrix in CHF ----')
rets_df, prices_df,  weights =risk_matrix.build_returns_matrix_in_chf(
                holdings,
                params=config.params,
                window_start=window_start,
                DEBUG=True)

print('weights\n', weights)

print('---- Calculating portfolio risk ----')
results = risk_matrix.portfolio_risk(rets_df, weights)

pf_vol = results["port_vol"]
summary_all = results["summary"]
print('full summary\n', summary_all)
summary = summary_all[~summary_all.index.str.startswith("CASH_")].copy()

print('summary\n', summary)
assets = summary.index

weights = summary["Weight"].astype(float)
print('weights:\n', weights)
mrc = summary["MRC"].astype(float)

desired_dd = 0.04

ohlc = True
fx_map = portfolio.make_fx_map(holdings, config.params, usd_shift=True, ohlc=ohlc)
high_df, low_df, close_df, close_local_df = portfolio.base_ccy_assets_px_df(holdings, fx_map, config.params, ohlc)

# remove cash from weights
w=weights[~pd.Index(assets).str.startswith("CASH_")]

# print('type assets:', type(assets))
# print('------len weights:', len(w), 'len assets:', len(assets))

# 1) COMPUTE ATR% (WILDER) FROM OHLC IN THE SAME CURRENCY BASIS AS WEIGHTS/RETURNS (CHF)
# EXPECT DATAFRAMES: HIGH_DF, LOW_DF, CLOSE_DF WITH COLUMNS MATCHING 'ASSETS'
high = high_df[assets]
low = low_df[assets]
close = close_df[assets]

# print('close_df\n', close_df.head())
# print('close\n', close.head())

prev_close = close.shift(1)
tr = pd.concat([
    (high - low),
    (high - prev_close).abs(),
    (low - prev_close).abs()
], axis=1).groupby(level=0, axis=1).max()  # MAX OF THE THREE TR COMPONENTS PER ASSET

atr = tr.ewm(alpha=1/14, adjust=False).mean()  
atr_pct = (atr.iloc[-1] / close.iloc[-1]).astype(float).clip(lower=1e-6) # Wilder's 14-period RMA

atr_pct = atr_pct.reindex(assets)
assert atr_pct.index.equals(assets)
# 2) PER-ASSET DD BUDGETS BY ABSOLUTE TRC (OR USE EQUAL SHARE)
trc_abs = (w * mrc).abs()
share = trc_abs / trc_abs.sum()
b = desired_dd * share  # sum(b) = desired_dd

# 3) L1 STOP DISTANCES AND ATR MULTIPLES
abs_w = w.abs().clip(lower=1e-12)


# MAP CATEGORIES: DEFAULT TACTICAL; OVERRIDE SPECIFIC TICKERS
cat = {  # examples; edit to your book
    'EMIM': 'core',
    'VUAG': 'core',
    'VEU': 'core',
    'ISJP': 'core',
    'IEMS': 'core',
    'XXSC': 'core',
    'IPLT': 'diversifier',
    'REMX': 'diversifier',
    'INRG': 'diversifier',
    "SGLN": "diversifier",
    # "YCA": "diversifier",   # set if you want it wider or excluded
}

cat_s = pd.Series({a: cat.get(a, "tactical") for a in assets}).reindex(assets)
defaults = [a for a in assets if a not in cat]
print("Asset categories:\n", cat_s)
print("Defaulted to 'tactical':", defaults)
print(cat_s.value_counts())
min_k_by_cat = {"core": 4.0, "tactical": 2, "diversifier": 5, "hedge": float("inf")}
include_in_stops = {"core": True, "tactical": True, "diversifier": True, "hedge": False}

min_k = cat_s.map(min_k_by_cat).astype(float)
take_stop = cat_s.map(include_in_stops).fillna(True).astype(bool)
# print("take stop:\n", take_stop)

min_pct, max_pct = 0.002, 0.10
# INITIAL L1 PROPOSAL
s = (b / abs_w).astype(float)

# APPLY CATEGORY FLOORS AND INCLUSION MASK
# print("min_k:\n", min_k)
# print("atr_pct:\n", atr_pct)
floor = ((min_k * atr_pct).astype(float)).clip(lower=min_pct)
# print('floor:\n', floor)
s.loc[~take_stop] = np.nan
s.loc[take_stop] = np.minimum(np.maximum(s.loc[take_stop].values, floor.loc[take_stop].values), max_pct)

# --- Feasibility-aware rescale ---
def _L1(total):
    return float((abs_w[take_stop] * total[take_stop]).sum())

L1_floor = _L1(floor)
if L1_floor > desired_dd + 1e-10:
    print(f"Infeasible: floors imply min DD {L1_floor:.4f} > target {desired_dd:.4f}. Using floors.")
    s.loc[take_stop] = floor.loc[take_stop]
else:
    # Rescale down while respecting floors (water-fill)
    def enforce_L1_with_floors(abs_w, s, floor, take_stop, dd, tol=1e-10, max_iter=20):
        s = s.copy()
        for _ in range(max_iter):
            L1 = float((abs_w[take_stop] * s[take_stop]).sum())
            if L1 <= dd + tol:
                break
            free = take_stop & (s > floor + 1e-12)
            if not free.any():
                break
            W = float(abs_w[free].sum())
            if W <= 0:
                break
            delta = (L1 - dd) / max(W, 1e-12)
            s.loc[free] = np.maximum(floor[free], s[free] - delta)
        return s

    s = enforce_L1_with_floors(abs_w, s, floor, take_stop, desired_dd)

# Final stops and diagnostics
stop_pct = s.astype(float)
# print('assets:\n', assets)
# print('stop_pct:\n', stop_pct)
# print('floor:\n', floor )
k_multiple = (stop_pct / atr_pct).astype(float)
dd_check = float((abs_w[take_stop] * stop_pct[take_stop]).sum())
print(f"L1 sum check ~ {dd_check:.4f} (target {desired_dd:.4f})")
binds = (take_stop & (stop_pct <= floor + 1e-12))
if binds.any():
    print("At floor for:", list(stop_pct[binds].index))

# 4) STOP PRICES
last_px_local = close_local_df[assets].iloc[-1].astype(float)

stop_px_local = pd.Series(
    np.where(w >= 0,
             last_px_local * (1.0 - stop_pct),
             last_px_local * (1.0 + stop_pct)),
    index=assets, name="stop_price_local"
)
last_px_chf = close.iloc[-1].astype(float).reindex(assets)
stop_px_chf = pd.Series(
    np.where(w >= 0, last_px_chf * (1.0 - stop_pct), last_px_chf * (1.0 + stop_pct)),
    index=assets, name="stop_price"
)

# 5)  L1 SUM CHECK STILL IN CHF BASIS: WORST-CASE SUM SHOULD HIT TARGET DD.
dd_check = float((abs_w[take_stop] * stop_pct[take_stop]).sum())
print(f"L1 sum check ~ {dd_check:.4f} (target {desired_dd:.4f})")

out = pd.DataFrame({
    "Weight": w,
    "ATR_pct": round(atr_pct*100,2),
    "stop_pct": round(stop_pct*100, 2),
    "k_ATR_mult": k_multiple,
    "last_px_local": last_px_local,
    "stop_px_local": stop_px_local,
    "last_px_chf": last_px_chf,
    "stop_px_chf": stop_px_chf
}).round(6)

print("Portfolio σ (annualized, CHF): ", pf_vol)

# Optionally drop cash/zero weights
# mask_live = (w.abs() > 1e-6) & ~pd.Index(assets).str.startswith("CASH_")
# print(out[mask_live])
print(out)

max age: 2
---- Building returns matrix in CHF ----
>>>FX keys loaded: ['GBPCHF', 'JPYCHF', 'USDCHF']
>>>Common window: 2021-09-27 → 2025-10-17 (480 rows before tail)
After alignment only 104 rows remain (expected 194 days 00:00:00). Data source may not have full history.
LOOKBACK DAYS/REGIME: 2025-04-10 00:00:00 to 2025-10-21 00:00:00  (194 days)
Total portfolio value (CHF): 1279583.25
VEU: value CHF6356.06,  last 72.22 *fx* 111
EMIM: value CHF650854.30,  last 3294.50 *fx* 187
VUAG: value CHF3828.60,  last 95.49 *fx* 38
ISJP: value CHF1250.82,  last 7648.00 *fx* 31
IEMS: value CHF643.93,  last 101.66 *fx* 6
XXSC: value CHF121248.04,  last 5756.00 *fx* 20
SGLN: value CHF275816.92,  last 6176.00 *fx* 42
IPLT: value CHF736.71,  last 24.57 *fx* 40
REMX: value CHF271.44,  last 14.42 *fx* 25
INRG: value CHF219453.30,  last 728.75 *fx* 290
CASH_CHF: value CHF8313.00,  last 1.00 *fx* 0.0
CASH_GBP: value CHF-43.65,  last 1.07 *fx* 0.0
CASH_USD: value CHF-7894.52,  last 0.79 *fx* 0.0
CASH_JPY: 

/var/folders/mp/2szq6ct92zl4hbdd0y18rqyw0000gn/T/ipykernel_93700/51634246.py:75: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  ], axis=1).groupby(level=0, axis=1).max()  # MAX OF THE THREE TR COMPONENTS PER ASSET


In [ ]:



trc = (weights * mrc).abs()
share = trc / trc.sum()  
# print('share', share)        # risk share by asset
desired_dd = 0.04
b = desired_dd * share 
# print('b',b)    

# STOP TERMS
abs_weigths = np.abs(weights) + 1e-18
stop_pct_L1 = b / abs_weigths

# BLEND AND ENFORCE L1 CAP
lam = 0.3
stop_pct = (1 - lam) * 

